# بخش طراحی: انتخاب Baseline و معیار مقایسه
## پروژه تشخیص و طبقه‌بندی صداهای محیطی با ESC-50

هدف این نوت‌بوک تعریف و ارزیابی دو خط مبنا است تا عملکرد مدل‌های بعدی با یک معیار ثابت مقایسه شود.

### Baselineها
1. **Majority-class baseline:** همیشه پرتکرارترین کلاسِ داده‌های آموزشی را پیش‌بینی می‌کند. این روش یک حداقل مرجع ساده است.
2. **MFCC + SVM:** از هر فایل صوتی ویژگی‌های MFCC استخراج می‌شود و یک طبقه‌بند SVM با کرنل RBF روی ویژگی‌ها آموزش می‌بیند.

### معیارهای ارزیابی
- Accuracy: نسبت پیش‌بینی‌های درست به کل نمونه‌ها
- Macro Precision، Macro Recall و Macro F1: میانگین بدون وزن معیارها بین کلاس‌ها؛ برای مسئله چندکلاسه مناسب‌اند.
- Confusion Matrix: نمایش کلاس‌هایی که مدل با یکدیگر اشتباه می‌گیرد.

ارزیابی با پنج Fold رسمی ESC-50 انجام می‌شود. در هر تکرار، یک Fold برای آزمون و چهار Fold برای آموزش استفاده می‌شود. ویژگی‌ها از فایل‌های صوتی استخراج می‌شوند و استانداردسازی فقط روی داده آموزشی هر Fold برازش می‌شود تا نشت اطلاعات رخ ندهد.

In [ ]:
# در صورت نیاز، وابستگی‌ها را نصب کنید:
# %pip install numpy pandas matplotlib seaborn scikit-learn librosa soundfile

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## ۱. معرفی مسیر داده‌ها

مجموعه ESC-50 را از مخزن رسمی دریافت و از حالت فشرده خارج کنید. مسیر پوشه‌ای را وارد کنید که درون آن پوشه‌های `audio` و فایل `meta/esc50.csv` وجود دارد.

ساختار مورد انتظار:
```text
ESC-50/
├── audio/
└── meta/
    └── esc50.csv
```

In [ ]:
# مسیر را متناسب با رایانه خود تغییر دهید.
DATA_ROOT = Path("data/ESC-50")

META_PATH = DATA_ROOT / "meta" / "esc50.csv"
AUDIO_DIR = DATA_ROOT / "audio"

if not META_PATH.exists():
    raise FileNotFoundError(
        f"فایل {META_PATH} پیدا نشد. ابتدا ESC-50 را دریافت و از حالت فشرده خارج کنید."
    )

meta = pd.read_csv(META_PATH)
meta.head()

In [ ]:
required_columns = {"filename", "fold", "target", "category"}
missing_columns = required_columns - set(meta.columns)
if missing_columns:
    raise ValueError(f"ستون‌های ضروری در metadata وجود ندارند: {missing_columns}")

meta["path"] = meta["filename"].apply(lambda name: AUDIO_DIR / name)
meta["file_exists"] = meta["path"].apply(lambda p: p.exists())

print("تعداد ردیف‌های metadata:", len(meta))
print("تعداد فایل‌های موجود:", int(meta["file_exists"].sum()))
print("تعداد فایل‌های مفقود:", int((~meta["file_exists"]).sum()))
print("تعداد کلاس‌ها:", meta["category"].nunique())
print("Foldها:", sorted(meta["fold"].unique()))
display(meta[["filename", "fold", "category", "file_exists"]].head())

## ۲. استخراج ویژگی‌های MFCC

برای هر فایل، ۱۳ ضریب MFCC محاسبه می‌شود. میانگین و انحراف معیار هر ضریب در طول زمان به‌عنوان ویژگی استفاده می‌شود؛ بنابراین برای هر فایل ۲۶ ویژگی خواهیم داشت.

اگر کتابخانه `librosa` نصب نیست، سلول نصب بالا را اجرا کنید.

In [ ]:
import librosa

def extract_mfcc_features(audio_path, sr=22050, n_mfcc=13):
    y, sample_rate = librosa.load(audio_path, sr=sr, mono=True)
    mfcc = librosa.feature.mfcc(y=y, sr=sample_rate, n_mfcc=n_mfcc)
    features = np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])
    return features

valid_meta = meta.loc[meta["file_exists"]].copy()
X_rows, y_rows, fold_rows, file_rows = [], [], [], []

for row in valid_meta.itertuples(index=False):
    try:
        X_rows.append(extract_mfcc_features(row.path))
        y_rows.append(row.category)
        fold_rows.append(int(row.fold))
        file_rows.append(row.filename)
    except Exception as exc:
        print(f"خطا در فایل {row.filename}: {exc}")

X = np.asarray(X_rows, dtype=np.float32)
y = np.asarray(y_rows)
folds = np.asarray(fold_rows)
filenames = np.asarray(file_rows)

print("شکل ماتریس ویژگی‌ها:", X.shape)
print("تعداد نمونه‌های قابل استفاده:", len(y))
print("تعداد کلاس‌ها:", len(np.unique(y)))

## ۳. تعریف مدل‌های مبنا

مدل اول، پیش‌بینی کلاس پرتکرار در داده آموزشی است. مدل دوم، SVM با کرنل RBF است. استانداردسازی ویژگی‌ها داخل Pipeline قرار دارد تا پارامترهای مقیاس‌بندی فقط از داده آموزشی هر Fold یاد گرفته شوند.

In [ ]:
models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "MFCC + SVM (RBF)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced")
    )
}

## ۴. ارزیابی با پنج Fold رسمی

در هر مرحله، یک Fold کامل آزمون و چهار Fold دیگر آموزش هستند. معیارها برای هر Fold و همچنین به‌صورت میانگین گزارش می‌شوند.

In [ ]:
results = []
all_predictions = {}

for model_name, model in models.items():
    y_true_all, y_pred_all = [], []

    for test_fold in sorted(np.unique(folds)):
        train_mask = folds != test_fold
        test_mask = folds == test_fold

        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y[train_mask], y[test_mask]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

        results.append({
            "model": model_name,
            "fold": int(test_fold),
            "accuracy": accuracy_score(y_test, y_pred),
            "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0)
        })

    all_predictions[model_name] = (np.asarray(y_true_all), np.asarray(y_pred_all))

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
summary = results_df.groupby("model")[[
    "accuracy", "macro_precision", "macro_recall", "macro_f1"
]].agg(["mean", "std"])

display(summary)

## ۵. نمودار مقایسه معیارها

In [ ]:
plot_df = results_df.melt(
    id_vars=["model", "fold"],
    value_vars=["accuracy", "macro_precision", "macro_recall", "macro_f1"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="metric", y="score", hue="model", errorbar="sd")
plt.ylim(0, 1)
plt.title("Baseline comparison across official ESC-50 folds")
plt.ylabel("Score")
plt.xlabel("Metric")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## ۶. ماتریس درهم‌ریختگی مدل MFCC + SVM

ماتریس درهم‌ریختگی نشان می‌دهد مدل کدام گروه‌های صوتی را با یکدیگر اشتباه می‌گیرد. برای خوانایی، برچسب کلاس‌ها در محورهای نمودار قرار می‌گیرند.

In [ ]:
svm_name = "MFCC + SVM (RBF)"
if svm_name in all_predictions:
    y_true, y_pred = all_predictions[svm_name]
    labels = sorted(np.unique(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title("Confusion Matrix - MFCC + SVM")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## ۷. گزارش متنی و ذخیره نتایج

اعداد نهایی فقط پس از اجرای کامل نوت‌بوک روی فایل‌های ESC-50 تولید می‌شوند. از ثبت عددهای فرضی خودداری کنید.

In [ ]:
output_dir = Path("outputs/baseline")
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "baseline_fold_results.csv", index=False)

summary_flat = results_df.groupby("model")[[
    "accuracy", "macro_precision", "macro_recall", "macro_f1"
]].agg(["mean", "std"])
summary_flat.to_csv(output_dir / "baseline_summary.csv")

print("نتایج در پوشه outputs/baseline ذخیره شدند.")
display(summary_flat)

## ۸. برداشت و نتیجه‌گیری

پس از اجرای نوت‌بوک، موارد زیر را با اعداد واقعی تکمیل کنید:

- عملکرد Majority baseline برابر با ... است.
- عملکرد MFCC + SVM از نظر Accuracy برابر با ... و از نظر Macro F1 برابر با ... است.
- اختلاف عملکرد دو روش نشان می‌دهد که ویژگی‌های صوتی MFCC تا چه اندازه به تفکیک کلاس‌ها کمک کرده‌اند.
- کلاس‌هایی که در ماتریس درهم‌ریختگی بیشترین اشتباه را دارند عبارت‌اند از: ...
- پراکندگی نتایج بین Foldها نشان‌دهنده میزان حساسیت مدل به تقسیم‌بندی داده‌ها است.

**معیار انتخاب مدل برای مراحل بعد:** Macro F1 معیار اصلی مقایسه در نظر گرفته می‌شود؛ زیرا به عملکرد همه کلاس‌ها وزن برابر می‌دهد. Accuracy نیز به‌عنوان معیار مکمل گزارش می‌شود.

### نکات روش‌شناختی
- این دو روش، Baseline هستند و الزاماً بهترین مدل ممکن نیستند.
- تنظیم پارامترهای SVM باید فقط با داده‌های آموزشی انجام شود؛ برای مقایسه اولیه، تنظیمات ثابت این نوت‌بوک استفاده شده‌اند.
- اگر فایل‌هایی از دیتاست موجود نباشند یا استخراج ویژگی برای آن‌ها شکست بخورد، تعداد نمونه‌های تحلیل‌شده کاهش می‌یابد؛ این موضوع باید در گزارش ذکر شود.